# Session 7 · Text Mining and Web Mining

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/01-data-mining-with-weka/code/session-07-text-mining-and-web-mining.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 4 · TF-IDF, classification and similarity in scikit-learn

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics.pairwise import cosine_similarity

pos = ["fast delivery and great quality", "excellent product works perfectly", "great price and fast shipping",
       "very happy with the quality", "excellent service will buy again", "perfect size and great colour",
       "happy customer fast response", "quality is excellent and price is fair", "great value highly recommended",
       "works great very happy"]
neg = ["late delivery and poor packaging", "product broke after one week", "poor quality waste of money",
       "delivery was very late", "item arrived damaged and broken", "refund still not received",
       "terrible service never again", "poor customer support and late refund", "damaged box and missing parts",
       "waste of money do not buy"]
texts, labels = pos + neg, ["pos"] * 10 + ["neg"] * 10

model = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2)), MultinomialNB())
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("5-fold accuracy:", cross_val_score(model, texts, labels, cv=cv).mean().round(3))

model.fit(texts, labels)
new = ["great quality and fast delivery", "box arrived broken, want a refund", "not happy at all"]
for t, p in zip(new, model.predict(new)):
    print(f"  {p}  <- {t}")

vec, nb = model.named_steps["tfidfvectorizer"], model.named_steps["multinomialnb"]
words = vec.get_feature_names_out()
diff = pd.Series(nb.feature_log_prob_[1] - nb.feature_log_prob_[0], index=words)  # classes: neg, pos
print("\nMost positive terms:", ", ".join(diff.sort_values().index[-6:][::-1]))
print("Most negative terms:", ", ".join(diff.sort_values().index[:6]))

X = vec.transform(texts)
query = vec.transform(["the delivery came late"])
sim = cosine_similarity(query, X).ravel()
print("\nReviews most similar to 'the delivery came late':")
for i in sim.argsort()[::-1][:3]:
    print(f"  {sim[i]:.2f}  {texts[i]}")

5-fold accuracy: 0.95
  pos  <- great quality and fast delivery
  neg  <- box arrived broken, want a refund
  pos  <- not happy at all

Most positive terms: great, happy, excellent, fast, works, price
Most negative terms: late, poor, refund, waste money, money, waste

Reviews most similar to 'the delivery came late':
  0.75  delivery was very late
  0.46  late delivery and poor packaging
  0.25  fast delivery and great quality


### Example 5 · Web structure mining: PageRank

In [2]:
import numpy as np

links = {
    "Home":       ["Admissions", "Programs", "News"],
    "Admissions": ["Home", "Fees"],
    "Programs":   ["Home", "DataScience", "Admissions"],
    "DataScience": ["Programs", "Admissions"],
    "News":       ["Home", "Programs"],
    "Fees":       ["Admissions"],
}
pages = list(links)
n, d = len(pages), 0.85
idx = {p: i for i, p in enumerate(pages)}
M = np.zeros((n, n))
for src, outs in links.items():
    for dst in outs:
        M[idx[dst], idx[src]] = 1 / len(outs)

pr = np.full(n, 1 / n)
for it in range(100):
    new = (1 - d) / n + d * M @ pr
    if np.abs(new - pr).sum() < 1e-8:
        break
    pr = new
print(f"Converged after {it} iterations\n")
for p, s in sorted(zip(pages, pr), key=lambda x: -x[1]):
    print(f"{p:<12}{s:.3f}  {'#' * int(s * 100)}")

Converged after 54 iterations

Admissions  0.294  #############################
Home        0.234  #######################
Programs    0.160  ################
Fees        0.150  ##############
News        0.091  #########
DataScience 0.070  #######


### Example 6 · Web usage mining: sessions and paths

In [3]:
import pandas as pd

log = pd.DataFrame([
    ("u1", "09:00", "/home"), ("u1", "09:01", "/programs"), ("u1", "09:03", "/programs/data-science"), ("u1", "09:06", "/admissions"),
    ("u2", "09:02", "/home"), ("u2", "09:02", "/fees"),
    ("u3", "09:05", "/news"), ("u3", "09:06", "/home"), ("u3", "09:07", "/programs"), ("u3", "09:09", "/programs/data-science"), ("u3", "09:12", "/admissions"), ("u3", "09:15", "/admissions/apply"),
    ("u4", "09:10", "/home"), ("u4", "09:11", "/programs"),
    ("u5", "09:20", "/programs/data-science"), ("u5", "09:24", "/admissions"), ("u5", "09:27", "/admissions/apply"),
], columns=["user", "time", "page"])

print("Page views:")
print(log["page"].value_counts().to_string())

entry = log.groupby("user")["page"].first().value_counts()
exitp = log.groupby("user")["page"].last().value_counts()
print("\nEntry pages:", entry.to_dict())
print("Exit pages: ", exitp.to_dict())

log["next"] = log.groupby("user")["page"].shift(-1)
paths = log.dropna().groupby(["page", "next"]).size().sort_values(ascending=False)
print("\nMost common page-to-page moves:")
print(paths.head(4).to_string())

applied = log.groupby("user")["page"].apply(lambda s: "/admissions/apply" in set(s))
print(f"\nConversion to application: {applied.mean():.0%} of {len(applied)} visitors")

Page views:
page
/home                     4
/programs                 3
/programs/data-science    3
/admissions               3
/admissions/apply         2
/fees                     1
/news                     1

Entry pages: {'/home': 3, '/news': 1, '/programs/data-science': 1}
Exit pages:  {'/admissions/apply': 2, '/admissions': 1, '/fees': 1, '/programs': 1}

Most common page-to-page moves:
page                    next                  
/home                   /programs                 3
/programs/data-science  /admissions               3
/admissions             /admissions/apply         2
/programs               /programs/data-science    2

Conversion to application: 40% of 5 visitors
